In [58]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix
import pandas as pd
import scipy
from scipy.io import loadmat
import torch
import mne
from sklearn.linear_model import LogisticRegression


In [59]:
import sys
import os

sys.path.append(r'C:\Users\torho\PycharmProjects\TranformerP300')

In [60]:
from preprocessing import normalize_epochs_1, concatenate_epochs_16

In [61]:
from generate_data import build_sequences_new, build_averaged_dataset


In [62]:
def threshold_sequence(seq, std_multiplier=2.0):
    seq = seq.copy()
    # порог = среднее * 2
    mean_abs = np.mean(np.abs(seq))
    threshold = mean_abs * std_multiplier

    # бинаризация: 1 там, где сигнал выше порога
    binary = (np.abs(seq) > threshold).astype(int)

    return binary


def flatten_for_logreg(sequences):
    return sequences.reshape(sequences.shape[0], -1)


def run_logreg(X_train, y_train, X_test, y_test, threshold_multiplier=None):
    X_train_proc = X_train.copy()
    X_test_proc = X_test.copy()

    if threshold_multiplier is not None:
        X_train_proc = np.array([threshold_sequence(seq, threshold_multiplier) for seq in X_train_proc])
        X_test_proc = np.array([threshold_sequence(seq, threshold_multiplier) for seq in X_test_proc])

    X_train_flat = flatten_for_logreg(X_train_proc)
    X_test_flat = flatten_for_logreg(X_test_proc)

    logreg = LogisticRegression(
        max_iter=1000,
        multi_class='multinomial',
        solver='lbfgs',
        random_state=42
    )
    logreg.fit(X_train_flat, y_train)

    y_pred = logreg.predict(X_test_flat)
    accuracy = accuracy_score(y_test, y_pred)

    return accuracy, logreg

In [63]:
data = scipy.io.loadmat('C:/Users/torho/P300_Transformer/data_kirasirova/S1901-P300_classic.mat')
epochs = data['epochs']
labels = data['labels'].flatten()

print(f"Эпох: {epochs.shape[0]}")
print(f"Target: {np.sum(labels==1)}, Non-target: {np.sum(labels==0)}")

Эпох: 6759
Target: 415, Non-target: 6344


In [64]:
epochs_norm = normalize_epochs_1(epochs)


In [65]:
# склеенные

real_sequences, real_targets = concatenate_epochs_16(epochs_norm, labels)
real_sequences.shape



(367, 16, 3, 250)

In [66]:
print(f"Классы {np.unique(real_targets)}")
np.bincount(real_targets)

Классы [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15]


array([ 9, 16, 24, 31, 32, 22, 22, 31, 27, 37, 21, 33, 26, 21, 10,  5])

In [67]:
# синтетика
ch_names = ['Fz', 'Cz', 'Pz']
sfreq = 250
info = mne.create_info(ch_names, sfreq, ch_types='eeg')
events = np.column_stack((np.arange(len(labels)), np.zeros(len(labels), int), labels))
event_id = {'non-target': 0, 'target': 1}
epochs_mne = mne.EpochsArray(epochs_norm, info, events=events, event_id=event_id)
epochs_mne.filter(l_freq=1, h_freq=15)

all_target = epochs_mne['target']
all_non_target = epochs_mne['non-target']

synth_sequences, synth_targets = build_sequences_new(
    all_target, all_non_target, n_classes=16, n_samples=2000
)
synth_sequences.shape


Not setting metadata
6759 matching events found
No baseline correction applied
0 projection items activated
Setting up band-pass filter from 1 - 15 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 15.00 Hz
- Upper transition bandwidth: 3.75 Hz (-6 dB cutoff frequency: 16.88 Hz)
- Filter length: 825 samples (3.300 s)



C:\Users\torho\AppData\Local\Temp\ipykernel_57604\3547420511.py:8: RuntimeWarning: filter_length (825) is longer than the signal (250), distortion is likely. Reduce filter length or filter a longer signal.
  epochs_mne.filter(l_freq=1, h_freq=15)


(2000, 16, 3, 250)

In [68]:

# усредненные

synth_avg, synth_avg_targets = build_averaged_dataset(
    synth_sequences, synth_targets, K=10, repeats=10
)
synth_avg.shape


(160, 16, 3, 250)

In [69]:
def check_logreg(name, X, y, threshold_multipliers=[None, 1.5, 2.0, 2.5, 3.0]):
    print(f"\n {name} ")
    
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    
    results = []
    
    for thresh in threshold_multipliers:
        if thresh is None:
            print(f"\nБез порога ")
        else:
            print(f"\nПорог = {thresh} ")
        
        acc, model = run_logreg(X_train, y_train, X_test, y_test, threshold_multiplier=thresh)
        results.append({'threshold': thresh, 'accuracy': acc * 100})
        print(f"Точность: {acc*100:.2f}%")
    
    return results

In [70]:
results_concat= check_logreg("Склеенные данные", real_sequences, real_targets)



 Склеенные данные 

Без порога 


C:\Users\torho\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Точность: 54.05%

Порог = 1.5 


C:\Users\torho\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Точность: 36.49%

Порог = 2.0 


...

Точность: 45.95%

Порог = 2.5 


C:\Users\torho\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Точность: 55.41%

Порог = 3.0 


C:\Users\torho\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Точность: 50.00%


In [71]:
results_synth = check_logreg("Синтетические", synth_sequences, synth_targets)



 Синтетические 

Без порога 


C:\Users\torho\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Точность: 42.50%

Порог = 1.5 


C:\Users\torho\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Точность: 27.25%

Порог = 2.0 


...

Точность: 35.50%

Порог = 2.5 


C:\Users\torho\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Точность: 43.00%

Порог = 3.0 


C:\Users\torho\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Точность: 45.50%


In [72]:
results_synth_avg = check_logreg("Усредненные синтетические", synth_avg, synth_avg_targets)




 Усредненные синтетические 

Без порога 


C:\Users\torho\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Точность: 100.00%

Порог = 1.5 


C:\Users\torho\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Точность: 84.38%

Порог = 2.0 


...

Точность: 96.88%

Порог = 2.5 


C:\Users\torho\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Точность: 100.00%

Порог = 3.0 


C:\Users\torho\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Точность: 100.00%
